In [ ]:
pip install  -u datasets transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import re
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer
import numpy as np

from transformers import (
    AutoModel,
)

from datasets import (
    Dataset,
    DatasetDict,
    Features, Sequence, ClassLabel, Value
)

In [ ]:
model_name =  "clicknext/phayathaibert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    text = text.strip()
    text = re.sub(r'http[s]?://\S+', ' ลิงก์ ', text)
    text = re.sub(r'\d+', '0', text)
    return text

def preprocess_phayathai(examples: dict) -> dict:
    return tokenizer(
        examples['clean_text'],
        padding='max_length',
        truncation=True,
        max_length= 130,
        return_tensors='pt'
    )   

In [4]:
train_df = pd.read_csv("train_data.csv")
val_df = pd.read_csv("val_data.csv")

train_df['clean_text'] = train_df['Text'].apply(clean_text)
val_df['clean_text'] = val_df['Text'].apply(clean_text)

NameError: name 'pd' is not defined

In [3]:
dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(train_df[['clean_text', 'Label']]),
    'validation': Dataset.from_pandas(val_df[['clean_text', 'Label']])
})

tokenized_datasets = dataset_dict.map(
    preprocess_phayathai, 
    batched=True,
    remove_columns=['clean_text']
)

tokenized_datasets.set_format('torch')
print(tokenized_datasets)

NameError: name 'Dataset' is not defined

In [2]:
sample_ids = tokenized_datasets['train'][8]['input_ids']
print(tokenizer.decode(sample_ids))

NameError: name 'tokenized_datasets' is not defined

Cross Validation

In [30]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import os

labels = np.array(tokenized_datasets['train']['Label'])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X=np.zeros(len(labels)), y=labels)):
    
    train_fold = tokenized_datasets['train'].select(train_idx)
    val_fold = tokenized_datasets['train'].select(val_idx)
    
    fold_ds_dict = DatasetDict({
        'train': train_fold,
        'validation': val_fold
    })
    
    print(fold_ds_dict)

    fold_ds_dict.save_to_disk("../model")
 

DatasetDict({
    train: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 432
    })
    validation: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 109
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 109/109 [00:00<00:00, 8436.75 examples/s] 


DatasetDict({
    train: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 433
    })
    validation: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 108
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 108/108 [00:00<00:00, 8125.58 examples/s]


DatasetDict({
    train: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 433
    })
    validation: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 108
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 108/108 [00:00<00:00, 6078.46 examples/s]


DatasetDict({
    train: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 433
    })
    validation: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 108
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 108/108 [00:00<00:00, 9003.51 examples/s] 


DatasetDict({
    train: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 433
    })
    validation: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 108
    })
})


Saving the dataset (1/1 shards): 100%|██████████| 108/108 [00:00<00:00, 10055.83 examples/s]
